# Part 2a — Open and look at LArCV input files

LArCV files (`.root`) are what SPINE **reads**: 3D images of the detector made of
**voxels** (3D pixels), plus, for simulation, the **true** labels and particles.
In this notebook you will:
1. list what is inside a LArCV file;
2. load one event with SPINE's LArCV reader (no neural network involved);
3. plot it in 3D, coloured by charge and by true particle type;
4. compare a simple **generic** event with a realistic **ProtoDUNE-SP** one, which has *ghost points*;
5. see the raw ROOT access underneath, and a container-free alternative.

**Session needed:** Jupyter inside the SPINE container (CPU is fine: partition `milano`, 0 GPUs).
Sections 1–4 need ROOT + LArCV, which only exist in the container.

> **Terminal equivalent:**
> ```bash
> spine_container.sh python -c "from spine.io.dataset import LArCVDataset as D; print(D.list_data('$TUTORIAL_DATA/generic_small.root'))"
> ```
> Or run this whole notebook non-interactively:
> `spine_container.sh jupyter nbconvert --to html --execute 02a_open_larcv_files.ipynb`

## 0. Setup

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import tutorial_env as te
env = te.setup()

generic_root = te.fetch("generic_small.root")
pdsp_root = te.fetch("protodune-sp_small.root")

## 1. What is inside a LArCV file?

A LArCV file is a ROOT file holding one **tree** per data product, named `<product>_tree`.
Each entry of a tree is one event. The prefix tells you what kind of product it is:

| Prefix | Contains | Example |
|---|---|---|
| `sparse3d_` | a list of voxels, each with one value | `sparse3d_pcluster` = deposited charge per voxel |
| `cluster3d_` | voxels grouped by the particle that made them (truth) | `cluster3d_pcluster` |
| `particle_` | list of true particles: PDG code, energy, start point… | `particle_corrected` |
| `neutrino_` | true neutrino interactions | `neutrino_mpv` |
| `opflash_`, `crthit_` | light flashes, cosmic-ray-tagger hits | |

`LArCVDataset.list_data` lists the products of a file:

In [ ]:
from spine.io.dataset import LArCVDataset

for path in (generic_root, pdsp_root):
    print(path.name)
    for kind, names in LArCVDataset.list_data(str(path)).items():
        print(f"   {kind:<10} {names}")

**What you should see:** the generic file has `sparse3d_pcluster`,
`sparse3d_pcluster_semantics`, `cluster3d_pcluster`, `particle_corrected`, … ProtoDUNE-SP has many
more, including `sparse3d_reco*` (reconstructed 3D points) and `..._semantics_ghost`.

## 2. Load one generic event with SPINE

SPINE reads LArCV files through a **schema**: a dictionary saying "make a product called X
using parser P on tree T". It's the same `schema:` you find in every SPINE config's `io` block.
Here we ask for four products:

In [ ]:
generic_schema = {
    # input image: voxel coordinates + deposited charge
    "data":      {"provider": "sparse3d", "sparse_event": "sparse3d_pcluster"},
    # true semantic label of every voxel (shower, track, Michel, delta, low energy)
    "seg_label": {"provider": "sparse3d", "sparse_event": "sparse3d_pcluster_semantics"},
    # list of true particles (positions in cm instead of voxel units)
    "particles": {"provider": "particle", "particle_event": "particle_corrected",
                  "cluster_event": "cluster3d_pcluster", "pixel_coordinates": False},
    # image geometry: where the voxel grid sits in the detector, voxel size
    "meta":      {"provider": "meta", "sparse_event": "sparse3d_pcluster"},
}

generic = LArCVDataset(file_keys=str(generic_root), schema=generic_schema, dtype="float32")
print(len(generic), "entries in the file")
event = generic[0]
print("products:", list(event.keys()))

`event["data"]` is a `TensorData`: `coords` holds the (N, 3) integer voxel indices and
`features` the (N, 1) charge. `meta` converts voxel indices to centimetres.

In [ ]:
data, meta = event["data"], event["meta"]
print("voxels:", data.coords.shape, " features:", data.features.shape)
print("voxel size [cm]:", meta.size, "  grid lower corner [cm]:", meta.lower)

xyz = meta.to_cm(data.coords, center=True)      # voxel index -> cm (voxel centre)
charge = data.features[:, 0]
print("first 3 voxels in cm:\n", xyz[:3])
print(f"total charge: {charge.sum():.1f}")

How many voxels of each true type? `SHAPE_LABELS` maps the label number to a name:

In [ ]:
from spine.constants import SHAPE_LABELS

labels = event["seg_label"].features[:, 0].astype(int)
values, counts = np.unique(labels, return_counts=True)
for v, c in zip(values, counts):
    print(f"{v}: {SHAPE_LABELS[v]:<8} {c:6d} voxels")

And the true particles (from the simulation):

In [ ]:
print(f"{'id':>3} {'pdg':>6} {'shape':<8} {'E_init [MeV]':>12} {'voxels':>7}   start [cm]")
for p in event["particles"]:
    if p.num_voxels <= 0:        # skip particles that left no visible charge
        continue
    print(f"{p.id:>3} {p.pdg_code:>6} {SHAPE_LABELS.get(p.shape, '?'):<8} "
          f"{p.energy_init:12.1f} {p.num_voxels:7d}   {np.round(p.position, 1)}")

## 3. Plot the event in 3D

`spine.vis.scatter_points` makes a plotly 3D scatter; `layout3d` sets sensible axes.
Drag to rotate, scroll to zoom, hover for values.

> If a figure is blank, run `import plotly.io as pio; pio.renderers.default = "iframe"` and re-run the cell.

In [ ]:
import plotly.graph_objs as go
from spine.vis import scatter_points, layout3d

fig = go.Figure(
    scatter_points(xyz, color=charge, markersize=2, colorscale="Inferno",
                   hovertext=[f"charge {q:.2f}" for q in charge], name="charge"),
    layout=layout3d(ranges=xyz, titles=["x [cm]", "y [cm]", "z [cm]"]),
)
fig.show()

In [ ]:
traces = []
for v in np.unique(labels):
    mask = labels == v
    traces += scatter_points(xyz[mask], markersize=2, name=SHAPE_LABELS[v])
fig = go.Figure(traces, layout=layout3d(ranges=xyz,
                                        titles=["x [cm]", "y [cm]", "z [cm]"]))
fig.show()

**What you should see:** tracks (long, thin) and showers (branchy). Click legend entries to hide or show each type.

## 4. A realistic detector: ProtoDUNE-SP and ghost points

In a wire detector like ProtoDUNE-SP, 3D points are *reconstructed* by combining the 2D
wire planes. Some combinations are wrong: these **ghost points** don't correspond to real
charge. Removing them is the first job of SPINE's network ("deghosting").

The input image is `sparse3d_reco`, and `sparse3d_pcluster_semantics_ghost` labels each
reconstructed point, with label 5 meaning *ghost*:

In [ ]:
pdsp_schema = {
    "data":      {"provider": "sparse3d", "sparse_event": "sparse3d_reco"},
    "seg_label": {"provider": "sparse3d", "sparse_event": "sparse3d_pcluster_semantics_ghost"},
    "meta":      {"provider": "meta", "sparse_event": "sparse3d_pcluster"},
}
pdsp = LArCVDataset(file_keys=str(pdsp_root), schema=pdsp_schema, dtype="float32")
print(len(pdsp), "entries")

ev = pdsp[0]
pxyz = ev["meta"].to_cm(ev["data"].coords, center=True)
plab = ev["seg_label"].features[:, 0].astype(int)
print(f"{len(pxyz)} reconstructed 3D points, {np.mean(plab == 5):.0%} of them are ghosts")

In [ ]:
# Plot at most 100k points so the browser stays responsive
rng = np.random.default_rng(0)
keep = rng.choice(len(pxyz), size=min(len(pxyz), 100_000), replace=False)

traces = []
for v in np.unique(plab):
    mask = plab[keep] == v
    traces += scatter_points(pxyz[keep][mask], markersize=1, name=SHAPE_LABELS.get(v, str(v)),
                             opacity=0.3 if v == 5 else 0.8)
fig = go.Figure(traces, layout=layout3d(ranges=pxyz,
                                        titles=["x [cm]", "y [cm]", "z [cm]"]))
fig.show()

**What you should see:** real tracks and showers surrounded by a haze of ghost points.
Toggle "Ghost" in the legend to see how much cleaner the true charge is.

## 5. Under the hood: raw ROOT access

SPINE's reader is a thin layer over ROOT. Each product `X` lives in tree `X_tree`, branch
`X_branch`, and LArCV provides helpers to copy voxels into numpy arrays. You rarely need this,
but it's useful when debugging a strange file.

In [ ]:
import ROOT
from larcv import larcv

chain = ROOT.TChain("sparse3d_pcluster_tree")
chain.AddFile(str(generic_root))
print("entries:", chain.GetEntries())
chain.GetEntry(0)
sparse_event = chain.sparse3d_pcluster_branch          # a larcv::EventSparseTensor3D

n = sparse_event.as_vector().size()
voxels = np.empty((n, 3), dtype=np.int32)
values = np.empty((n, 1), dtype=np.float32)
larcv.fill_3d_voxels(sparse_event, voxels)
larcv.fill_3d_pcloud(sparse_event, values)
print(n, "voxels, total charge", values.sum())
print("same as SPINE's reader:", n == len(data.coords) and np.isclose(values.sum(), charge.sum()))

## 6. Optional: no container? Use `uproot`

`uproot` reads ROOT files in pure Python, so it works on a laptop without ROOT or LArCV
(`pip install uproot`). Each voxel is stored as an `_id` (a flattened index into the grid)
plus a `_value`. With the grid description (`_meta`) we can turn ids back into centimetres.

In [ ]:
try:
    import uproot
except ImportError:
    uproot = None
    print("uproot is not installed here; skipping (pip install uproot to try it).")

if uproot is not None:
    branch = uproot.open(generic_root)["sparse3d_pcluster_tree/sparse3d_pcluster_branch"]
    cols = ["_voxel_v._id", "_voxel_v._value", "_meta._p1.x", "_meta._p1.y", "_meta._p1.z",
            "_meta._xlen", "_meta._ylen", "_meta._zlen", "_meta._xnum", "_meta._ynum"]
    a = branch.arrays(cols, library="np", entry_stop=1)          # first event only

    ids = a["_voxel_v._id"][0].astype(np.int64)
    nx, ny = a["_meta._xnum"][0], a["_meta._ynum"][0]
    ijk = np.stack([ids % nx, (ids // nx) % ny, ids // (nx * ny)], axis=1)   # id -> (i, j, k)
    lower = np.array([a["_meta._p1.x"][0], a["_meta._p1.y"][0], a["_meta._p1.z"][0]])
    size = np.array([a["_meta._xlen"][0], a["_meta._ylen"][0], a["_meta._zlen"][0]])  # voxel size [cm]
    xyz_uproot = lower + (ijk + 0.5) * size

    print(len(ids), "voxels; voxel size", size, "cm")
    print(f"total charge {a['_voxel_v._value'][0].sum():.1f}")
    print("x range [cm]:", xyz_uproot[:, 0].min().round(2), "to", xyz_uproot[:, 0].max().round(2))

## Summary

| Task | How |
|---|---|
| list products in a file | `LArCVDataset.list_data(path)` |
| load events as numpy | `LArCVDataset(file_keys=path, schema={...}, dtype="float32")[i]` |
| voxel indices → cm | `event["meta"].to_cm(event["data"].coords, center=True)` |
| label names | `from spine.constants import SHAPE_LABELS, PID_LABELS` |
| plot | `spine.vis.scatter_points` + `layout3d` |

The `schema` you wrote here is exactly what goes under `io: loader: dataset: schema:` in a SPINE
config, for both inference (Part 1) and training (Part 3).

Next: [`02b_analyze_spine_output.ipynb`](02b_analyze_spine_output.ipynb), what SPINE *made* from these inputs.